# Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells Exploration with `mlcroissant`
This notebook provides a step-by-step guide for loading, exploring, and analyzing the FAIR^2 mass spectrometry dataset using the `mlcroissant` library.

### Dataset Source
The dataset source is provided via a Croissant schema URL:

```
https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json
```

All record sets, fields, and columns are referenced using their `@id` as defined in the Croissant schema.

In [ ]:
# Ensure `mlcroissant` library is installed (upgrade for latest spec compliance)
!pip install --upgrade mlcroissant

## 1. Data Loading
Load metadata and dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd
import json

# Define the Croissant metadata URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json'

# Load the dataset
dataset = mlc.Dataset(croissant_url)
# Access metadata as an object (not via dict subscripting)
md = dataset.metadata
print(f"{md.name}\n---\n{md.description}\n")
print(f"License: {md.license}")
print(f"Published: {md.date_published}")


## 2. Data Overview
Review available record sets, fields, and their `@id`.

You should always reference entities (record sets, fields, columns) by their `@id`. Let's enumerate the record sets and the fields present in each.

In [ ]:
# List all record sets and their respective fields & field @ids
from pprint import pprint

print("Available Record Sets:\n----------------------")
for rs in md.record_sets:
    print(f"@id: {rs.id}")
    print(f"  Name: {rs.name}")
    print("  Fields:")
    for f in rs.fields:
        print(f"    @id: {f.id}\tName: {f.name}")
    print()

To get an idea of the sample records, let's preview the first record for each record set by its `@id`.

**Example:**

In [ ]:
for rs in md.record_sets:
    print(f"--- Record Set: {rs.name} (@id={rs.id}) ---")
    records = list(dataset.records(record_set=rs.id))
    if records:
        pprint(records[0])
        print(f"Keys: {records[0].keys()}\n")
    else:
        print("No records found in this record set.\n")

## 3. Data Extraction
Let's extract all data from the main quantitative record set for downstream processing.

Replace `<record_set_id>` with the `@id` of the record set you want to analyze. For illustrative purposes, we'll use the main protein table, likely named 'Protein Quantitative Table' or similar. Adjust `main_record_set_id` if your schema has a different name.

In [ ]:
# Find the main protein quantitation record set id
main_record_set_id = None
for rs in md.record_sets:
    # Choose the main data table (adjust name/logic as needed)
    if ('quant' in rs.name.lower()) or ('protein' in rs.name.lower() and 'table' in rs.name.lower()):
        main_record_set_id = rs.id
        break

assert main_record_set_id is not None, "Main record set not found. Check the schema names."
print(f"Main record set @id selected: {main_record_set_id}")

# For this notebook, let's extract all record set IDs
record_set_ids = [rs.id for rs in md.record_sets]
dataframes = {}
for rsid in record_set_ids:
    records = list(dataset.records(record_set=rsid))
    dataframes[rsid] = pd.DataFrame(records)

# Examine columns in the main table
print(dataframes[main_record_set_id].columns.tolist())
dataframes[main_record_set_id].head()

## 4. Exploratory Data Analysis (EDA)
Let's select a numeric field for analysis. All processing will be referenced by the *field's* `@id` (not its label).

First, we'll list all numeric fields in the main record set.

In [ ]:
# Identify numeric fields by their @id (schema:Float/Integer etc.) in the main table
numeric_field_ids = []
for rs in md.record_sets:
    if rs.id == main_record_set_id:
        for f in rs.fields:
            # Only common primitive numeric types
            if getattr(f, 'data_type', None) is not None and (
                ('Float' in f.data_type) or ('Integer' in f.data_type) or ('Number' in f.data_type)
            ):
                numeric_field_ids.append(f.id)
        break
print("Numeric field @ids in main table:")
print(numeric_field_ids)
# Choose a field for EDA. If present, pick percent_coverage or abundance.
chosen_numeric_field = None
for fid in numeric_field_ids:
    if 'abundance' in fid.lower():
        chosen_numeric_field = fid
        break
if not chosen_numeric_field and numeric_field_ids:
    chosen_numeric_field = numeric_field_ids[0] # fallback
print(f"Selected for analysis: {chosen_numeric_field}")

In [ ]:
# Filtering and normalization using @id columns
df = dataframes[main_record_set_id]
# Drop NA for safe filtering
col = chosen_numeric_field
df_numeric = df.copy()
df_numeric = df_numeric[df_numeric[col].notna()]

# Apply filter: values > threshold
threshold = df_numeric[col].mean() if df_numeric[col].dtype != 'O' else 10

filtered_df = df_numeric[df_numeric[col] > threshold]
print(f"Filtered records with {col} > {threshold:.2f}:")
print(filtered_df.head())

# Normalize the numeric field
filtered_df[f"{col}_normalized"] = (filtered_df[col] - filtered_df[col].mean()) / filtered_df[col].std()
print(f"Normalized {col} for filtered records:")
print(filtered_df[[col, f"{col}_normalized"]].head())

# Try grouping by a categorical field, such as 'gene_symbol' or similar
group_field = None
possible_groupers = [f.id for f in md.record_sets[0].fields if 'gene' in f.id.lower() or 'accession' in f.id.lower() or 'category' in f.id.lower()]
for g in possible_groupers:
    if g in filtered_df.columns:
        group_field = g
        break
if group_field:
    grouped_df = filtered_df.groupby(group_field).mean(numeric_only=True)
    print(f"Grouped data by {group_field}:")
    print(grouped_df[[col]].head())

## 5. Visualization
Visualize the distribution of the selected numeric field and the relationship with a grouping field, if chosen.

We'll use `matplotlib` and `seaborn` for plotting.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

plt.figure(figsize=(6, 4))
ax = sns.histplot(df_numeric[chosen_numeric_field], kde=True)
ax.set_title(f"Distribution of {chosen_numeric_field}")
ax.set_xlabel(chosen_numeric_field)
plt.tight_layout()
plt.show()

# If a group/categorical field is present, plot mean value per group
if group_field and group_field in grouped_df.index:
    topn = grouped_df[chosen_numeric_field].sort_values(ascending=False).head(20)
    plt.figure(figsize=(8, 6))
    sns.barplot(y=topn.index, x=topn.values, orient='h')
    plt.title(f"Top 20 groups by mean {chosen_numeric_field}")
    plt.xlabel(chosen_numeric_field)
    plt.ylabel(group_field)
    plt.tight_layout()
    plt.show()

## 6. Conclusion
In this notebook, we've demonstrated how to load and explore a FAIR^2-compliant mass spectrometry protein dataset using the `mlcroissant` library:

- All record sets, fields, and columns were referenced by their `@id`, ensuring schema-driven reproducibility.
- The notebook showed steps for programmatic metadata and data exploration, filtering, normalization, and visualization in a reproducible pipeline.
  
This workflow enables reliable downstream statistical and biological analysis for proteomic datasets described with a Croissant schema.